# Detecção e Reconstrução de Malha Viária a partir de Imagens de Satélite

**Disciplina:** Reconhecimento de Padrões  
**Integrantes:** Beatriz Vocurca Frade, Johnatan Augusto Moreira do Carmo e Marina Alves Resende

Este notebook implementa um pipeline de visão computacional clássica para detectar vias, gerar máscaras segmentadas, sobrepor o resultado na imagem original e extrair um grafo da malha viária.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "data" / "results"
GRAPHS_DIR = PROJECT_ROOT / "data" / "graphs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz: {PROJECT_ROOT}")


## 1. Pipeline de detecção

In [ ]:
from src.malha_viaria import (
    RoadDetectionResult,
    list_images,
    plot_all_results,
    process_dataset,
)

print("Pipeline carregado a partir de src/malha_viaria.py")
print(f"Imagens encontradas: {len(list_images(RAW_DIR))}")


## 2. Como usar

1. Coloque imagens de satélite em `data/raw/`.
2. Execute todas as células deste notebook.
3. As imagens de saída serão salvas em `data/results/`.
4. Os grafos serão exportados em `data/graphs/`, nos formatos JSON e GraphML.

Quando `data/raw/` estiver vazia, o notebook usa uma imagem sintética apenas para demonstrar que o pipeline está funcionando.


## 3. Ideia da solução

A segmentação combina normalização, CLAHE, suavização bilateral, descritores simples de cor, textura local, bordas e morfologia matemática. Depois, a máscara é esqueletonizada para extrair a topologia: pixels com grau 1 viram extremidades, pixels com grau maior ou igual a 3 viram interseções, e os caminhos entre esses vértices viram arestas do grafo.


## 4. Processamento em lote

A célula abaixo processa todas as imagens encontradas em `data/raw/`. O parâmetro `max_side` limita o maior lado da imagem para reduzir custo computacional, mantendo a proporção original.


In [ ]:
results, summary = process_dataset(
    raw_dir=RAW_DIR,
    results_dir=RESULTS_DIR,
    graphs_dir=GRAPHS_DIR,
    max_side=1400,
    demo_if_empty=True,
)

summary


## 5. Visualização dos resultados

Para cada imagem, são exibidos: imagem original, máscara binária, classes de via, sobreposição, esqueleto e grafo extraído.


In [ ]:
plot_all_results(results)

## 6. Arquivos gerados

Para cada imagem processada, o pipeline salva:

- `*_segmentada.png`: vias coloridas por classe;
- `*_sobreposicao.png`: vias sobrepostas à imagem original;
- `*_grafo.png`: grafo desenhado sobre a sobreposição;
- `*_mascara.png`: máscara binária;
- `*_esqueleto.png`: esqueleto da máscara;
- `*_grafo.json`: nós, arestas e pontos de cada segmento;
- `*_grafo.graphml`: grafo para ferramentas como Gephi, Cytoscape ou NetworkX.


In [ ]:
print("Resultados em:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file() and p.name != ".gitkeep":
        print(" -", p.name)
print("Grafos em:", GRAPHS_DIR)
for p in sorted(GRAPHS_DIR.glob("*")):
    if p.is_file() and p.name != ".gitkeep":
        print(" -", p.name)


## 7. Interpretação e limitações

A solução é propositalmente explicável. Ela tende a funcionar melhor quando as vias têm textura homogênea e contraste razoável em relação ao entorno. Os principais erros esperados são falsos positivos em telhados claros, solo exposto, sombras lineares e regiões agrícolas; falsos negativos podem ocorrer em vias muito estreitas, arborizadas ou com pavimento visualmente parecido com o ambiente.
